# Kami menggunakan sort berdasarkan medali "emas" 

dengan urutan medali (emas → perak → perunggu → harapan → partisipan)

# BUBBLE SORT

In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# Nama file dataset
file_path = "osn.csv"

# Load dataset
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)

# Mengambil kolom
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

# Mengisi data kosong
df["Medali"] = df["Medali"].fillna("Partisipan")

# =========================
# PRIORITAS MEDALI
# =========================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada medali lain yang tidak dikenal
df["Prioritas"] = df["Prioritas"].fillna(99)

# Mengubah dataframe menjadi list
data = df.values.tolist()

# =========================
# BUBBLE SORT BERDASARKAN MEDALI
# =========================
n = len(data)

for i in range(n):
    for j in range(0, n - i - 1):

        # Index 4 = Prioritas
        if data[j][4] > data[j + 1][4]:

            # Tukar posisi
            data[j], data[j + 1] = data[j + 1], data[j]

# Mengembalikan ke DataFrame
df_sorted = pd.DataFrame(
    data,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

# Pengaturan tampilan
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Menampilkan hasil
display(df_sorted)

# hasil eksekusi tercantum

In [ ]:
import pandas as pd
import kagglehub
import time
import threading
import tracemalloc

from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# =========================================================
# NAMA FILE DATASET
# =========================================================
file_path = "osn.csv"

# =========================================================
# LOAD DATASET
# =========================================================
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)

# =========================================================
# MENGAMBIL KOLOM
# =========================================================
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

# =========================================================
# MENGISI DATA KOSONG
# =========================================================
df["Medali"] = df["Medali"].fillna("Partisipan")

# =========================================================
# PRIORITAS MEDALI
# =========================================================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada medali lain yang tidak dikenal
df["Prioritas"] = df["Prioritas"].fillna(99)

# =========================================================
# MENGUBAH DATAFRAME MENJADI LIST
# =========================================================
data = df.values.tolist()

# =========================================================
# FUNGSI BUBBLE SORT
# =========================================================
def bubble_sort(arr):

    n = len(arr)

    for i in range(n):
        for j in range(0, n - i - 1):

            # Index 4 = Prioritas
            if arr[j][4] > arr[j + 1][4]:

                # Tukar posisi
                arr[j], arr[j + 1] = arr[j + 1], arr[j]

    return arr

# =========================================================
# SINGLE-THREAD EXECUTION
# =========================================================
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = bubble_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# =========================================================
# MULTI-THREAD EXECUTION
# =========================================================
multi_data = data.copy()

# Membagi data menjadi 2 bagian
mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

# Thread Bubble Sort
thread1 = threading.Thread(target=bubble_sort, args=(part1,))
thread2 = threading.Thread(target=bubble_sort, args=(part2,))

tracemalloc.start()

start_multi = time.time()

# Menjalankan thread
thread1.start()
thread2.start()

# Menunggu thread selesai
thread1.join()
thread2.join()

# Menggabungkan hasil
merged = part1 + part2

# Sort akhir
multi_sorted = bubble_sort(merged)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

# =========================================================
# PERFORMANCE IMPROVEMENT
# =========================================================
improvement = (
    (single_time - multi_time) / single_time
) * 100

# =========================================================
# MEMORY USAGE
# =========================================================
memory_usage_mb = peak_multi / (1024 * 1024)

# =========================================================
# DATAFRAME HASIL SORTING
# =========================================================
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

# =========================================================
# PENGATURAN TAMPILAN
# =========================================================
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =========================================================
# MENAMPILKAN HASIL SORTING
# =========================================================
display(df_sorted)

# =========================================================
# TABEL PERFORMA
# =========================================================
performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Bubble Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 4)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 4)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 2)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")

display(performance_df)

 # bubble dengan data kecil & sedang 

In [ ]:
import pandas as pd
import time
import threading
import tracemalloc
from IPython.display import display

# NAMA FILE CSV
file_path = "dataKecil.csv"
# file_path = "dataSeddang.csv"


df = pd.read_csv("dataKecil.csv", sep=";") # menggunakan separator ;

df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

df["Medali"] = df["Medali"].fillna("Partisipan") # mengisi data kosong dengan "Partisipan"

prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada nilai medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)
data = df.values.tolist()

def bubble_sort(arr):

    n = len(arr)

    for i in range(n):

        swapped = False

        for j in range(0, n - i - 1):

            # Membandingkan berdasarkan Prioritas (index 4)
            if arr[j][4] > arr[j + 1][4]:

                arr[j], arr[j + 1] = arr[j + 1], arr[j]

                swapped = True

        # Jika tidak ada pertukaran, data sudah terurut
        if not swapped:
            break

    return arr

# SINGLE-THREAD EXECUTION
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = bubble_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# MULTI-THREAD EXECUTION
multi_data = data.copy()

mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

hasil1 = []
hasil2 = []

# FUNGSI THREAD
def sort_part1():
    global hasil1
    hasil1 = bubble_sort(part1)

def sort_part2():
    global hasil2
    hasil2 = bubble_sort(part2)

thread1 = threading.Thread(target=sort_part1)
thread2 = threading.Thread(target=sort_part2)

tracemalloc.start()

start_multi = time.time()

thread1.start()
thread2.start()

thread1.join()
thread2.join()

merged = hasil1 + hasil2 # MENGGABUNGKAN HASIL

# Sorting akhir
multi_sorted = bubble_sort(merged)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

improvement = (
    (single_time - multi_time) / single_time
) * 100


memory_usage_mb = peak_multi / (1024 * 1024) # rumus untuk mengubah byte ke megabyte
                                                # Memory Usage (MB) = Peak Memory (Byte) 1024 * 1024

# HASIL SORTING
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nHASIL SORTING BUBBLE SORT:\n")
display(df_sorted)

performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Bubble Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 6)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 6)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 4)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")
display(performance_df)

# MERGE SORT

In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# Nama file dataset
file_path = "osn.csv"

# Load dataset
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)

# Mengambil kolom yang ingin ditampilkan
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

# Mengisi data kosong
df["Medali"] = df["Medali"].fillna("Partisipan")

# =========================
# PRIORITAS MEDALI
# =========================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)

# Mengubah dataframe menjadi list
data = df.values.tolist()

# =========================
# MERGE SORT
# =========================
def merge_sort(arr):

    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)

# =========================
# MERGE FUNCTION
# =========================
def merge(left, right):

    hasil = []

    while len(left) > 0 and len(right) > 0:

        # Index 4 = Prioritas Medali
        if left[0][4] < right[0][4]:

            hasil.append(left.pop(0))

        else:

            hasil.append(right.pop(0))

    # Sisa data
    hasil.extend(left)
    hasil.extend(right)

    return hasil

# Menjalankan Merge Sort
sorted_data = merge_sort(data)

# Mengembalikan ke DataFrame
df_sorted = pd.DataFrame(
    sorted_data,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

# Pengaturan tampilan
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Menampilkan hasil
display(df_sorted)

# hasil ekskusi tercantum

In [ ]:
import pandas as pd
import kagglehub
import time
import threading
import tracemalloc

from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# =========================================================
# NAMA FILE DATASET
# =========================================================
file_path = "osn.csv"

# =========================================================
# LOAD DATASET
# =========================================================
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)

# =========================================================
# MENGAMBIL KOLOM
# =========================================================
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

# =========================================================
# MENGISI DATA KOSONG
# =========================================================
df["Medali"] = df["Medali"].fillna("Partisipan")

# =========================================================
# PRIORITAS MEDALI
# =========================================================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)

# =========================================================
# MENGUBAH DATAFRAME MENJADI LIST
# =========================================================
data = df.values.tolist()

# =========================================================
# MERGE SORT
# =========================================================
def merge_sort(arr):

    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)

# =========================================================
# MERGE FUNCTION
# =========================================================
def merge(left, right):

    hasil = []

    while len(left) > 0 and len(right) > 0:

        # Index 4 = Prioritas Medali
        if left[0][4] < right[0][4]:

            hasil.append(left.pop(0))

        else:

            hasil.append(right.pop(0))

    # Sisa data
    hasil.extend(left)
    hasil.extend(right)

    return hasil

# =========================================================
# SINGLE-THREAD EXECUTION
# =========================================================
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = merge_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# =========================================================
# MULTI-THREAD EXECUTION
# =========================================================
multi_data = data.copy()

mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

hasil1 = []
hasil2 = []

# Fungsi thread
def sort_part1():
    global hasil1
    hasil1 = merge_sort(part1)

def sort_part2():
    global hasil2
    hasil2 = merge_sort(part2)

thread1 = threading.Thread(target=sort_part1)
thread2 = threading.Thread(target=sort_part2)

tracemalloc.start()

start_multi = time.time()

# Menjalankan thread
thread1.start()
thread2.start()

# Menunggu thread selesai
thread1.join()
thread2.join()

# Menggabungkan hasil
multi_sorted = merge(hasil1, hasil2)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

# =========================================================
# PERFORMANCE IMPROVEMENT
# =========================================================
improvement = (
    (single_time - multi_time) / single_time
) * 100

# =========================================================
# MEMORY USAGE
# =========================================================
memory_usage_mb = peak_multi / (1024 * 1024)

# =========================================================
# HASIL SORTING
# =========================================================
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

# =========================================================
# PENGATURAN TAMPILAN
# =========================================================
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =========================================================
# MENAMPILKAN HASIL SORTING
# =========================================================
display(df_sorted)

# =========================================================
# TABEL PERFORMA
# =========================================================
performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Merge Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 4)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 4)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 2)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")

display(performance_df)

 # merge data kecil & sedang

In [ ]:
import pandas as pd
import time
import threading
import tracemalloc
from IPython.display import display

# NAMA FILE CSV
file_path = "dataKecil.csv"
# file_path = "dataSeddang.csv"


df = pd.read_csv("dataKecil.csv", sep=";") # menggunakan separator ;

df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

df["Medali"] = df["Medali"].fillna("Partisipan") # mengisi data kosong dengan "Partisipan"

prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada nilai medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)
data = df.values.tolist()

def merge_sort(arr):

    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)


def merge(left, right):

    hasil = []

    i = 0
    j = 0

    # Membandingkan berdasarkan kolom Prioritas (index 4)
    while i < len(left) and j < len(right):

        if left[i][4] <= right[j][4]:
            hasil.append(left[i])
            i += 1
        else:
            hasil.append(right[j])
            j += 1

    hasil.extend(left[i:])
    hasil.extend(right[j:])

    return hasil

# SINGLE-THREAD EXECUTION
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = merge_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# MULTI-THREAD EXECUTION
multi_data = data.copy()

mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

hasil1 = []
hasil2 = []

# FUNGSI THREAD
def sort_part1():
    global hasil1
    hasil1 = merge_sort(part1)

def sort_part2():
    global hasil2
    hasil2 = merge_sort(part2)

thread1 = threading.Thread(target=sort_part1)
thread2 = threading.Thread(target=sort_part2)

tracemalloc.start()

start_multi = time.time()

thread1.start()
thread2.start()

thread1.join()
thread2.join()

merged = hasil1 + hasil2 # MENGGABUNGKAN HASIL

# Sorting akhir
multi_sorted = merge_sort(merged)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

improvement = (
    (single_time - multi_time) / single_time
) * 100


memory_usage_mb = peak_multi / (1024 * 1024) # rumus untuk mengubah byte ke megabyte
                                                # Memory Usage (MB) = Peak Memory (Byte) 1024 * 1024

# HASIL SORTING
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nHASIL SORTING MERGE SORT:\n")
display(df_sorted)

performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Merge Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 6)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 6)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 4)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")
display(performance_df)

# SHELL SORT

In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# Nama file dataset
file_path = "osn.csv"

# Load dataset
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)

# Mengambil kolom yang ingin ditampilkan
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

# Mengisi data kosong
df["Medali"] = df["Medali"].fillna("Partisipan")

# =========================
# PRIORITAS MEDALI
# =========================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)

# Mengubah dataframe menjadi list
data = df.values.tolist()

# =========================
# SHELL SORT
# =========================
n = len(data)

# Menentukan gap awal
gap = n // 2

while gap > 0:

    for i in range(gap, n):

        temp = data[i]
        j = i

        # Index 4 = Prioritas Medali
        while j >= gap and data[j - gap][4] > temp[4]:

            data[j] = data[j - gap]
            j -= gap

        data[j] = temp

    # Mengurangi gap
    gap //= 2

# Mengembalikan ke DataFrame
df_sorted = pd.DataFrame(
    data,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

# Pengaturan tampilan
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Menampilkan hasil
display(df_sorted)

# hasil eksekusi tercantum

In [ ]:
import pandas as pd
import kagglehub
import time
import threading
import tracemalloc

from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# =========================================================
# NAMA FILE DATASET
# =========================================================
file_path = "osn.csv"

# =========================================================
# LOAD DATASET
# =========================================================
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)

# =========================================================
# MENGAMBIL KOLOM
# =========================================================
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

# =========================================================
# MENGISI DATA KOSONG
# =========================================================
df["Medali"] = df["Medali"].fillna("Partisipan")

# =========================================================
# PRIORITAS MEDALI
# =========================================================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)

# =========================================================
# MENGUBAH DATAFRAME MENJADI LIST
# =========================================================
data = df.values.tolist()

# =========================================================
# FUNGSI SHELL SORT
# =========================================================
def shell_sort(arr):

    n = len(arr)

    # Menentukan gap awal
    gap = n // 2

    while gap > 0:

        for i in range(gap, n):

            temp = arr[i]
            j = i

            # Index 4 = Prioritas Medali
            while j >= gap and arr[j - gap][4] > temp[4]:

                arr[j] = arr[j - gap]
                j -= gap

            arr[j] = temp

        # Mengurangi gap
        gap //= 2

    return arr

# =========================================================
# SINGLE-THREAD EXECUTION
# =========================================================
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = shell_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# =========================================================
# MULTI-THREAD EXECUTION
# =========================================================
multi_data = data.copy()

# Membagi data menjadi 2 bagian
mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

hasil1 = []
hasil2 = []

# Fungsi thread
def sort_part1():
    global hasil1
    hasil1 = shell_sort(part1)

def sort_part2():
    global hasil2
    hasil2 = shell_sort(part2)

thread1 = threading.Thread(target=sort_part1)
thread2 = threading.Thread(target=sort_part2)

tracemalloc.start()

start_multi = time.time()

# Menjalankan thread
thread1.start()
thread2.start()

# Menunggu thread selesai
thread1.join()
thread2.join()

# Menggabungkan hasil
merged = hasil1 + hasil2

# Sorting akhir
multi_sorted = shell_sort(merged)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

# =========================================================
# PERFORMANCE IMPROVEMENT
# =========================================================
improvement = (
    (single_time - multi_time) / single_time
) * 100

# =========================================================
# MEMORY USAGE
# =========================================================
memory_usage_mb = peak_multi / (1024 * 1024)

# =========================================================
# HASIL SORTING
# =========================================================
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

# =========================================================
# PENGATURAN TAMPILAN
# =========================================================
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =========================================================
# MENAMPILKAN HASIL SORTING
# =========================================================
display(df_sorted)

# =========================================================
# TABEL PERFORMA
# =========================================================
performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Shell Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 4)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 4)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 2)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")

display(performance_df)

# shell data kecil dan sedang

In [ ]:
import pandas as pd
import time
import threading
import tracemalloc
from IPython.display import display

# NAMA FILE CSV
# file_path = "dataKecil.csv"
file_path = "dataSeddang.csv"


df = pd.read_csv("dataKecil.csv", sep=";") # menggunakan separator ;

df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

df["Medali"] = df["Medali"].fillna("Partisipan") # mengisi data kosong dengan "Partisipan"

prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada nilai medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)
data = df.values.tolist()

def shell_sort(arr):

    n = len(arr)
    gap = n // 2

    while gap > 0:

        for i in range(gap, n):

            temp = arr[i]
            j = i

            while j >= gap and arr[j - gap][4] > temp[4]:

                arr[j] = arr[j - gap]
                j -= gap

            arr[j] = temp

        gap //= 2

    return arr

# SINGLE-THREAD EXECUTION
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = shell_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# MULTI-THREAD EXECUTION
multi_data = data.copy()

mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

hasil1 = []
hasil2 = []

# FUNGSI THREAD
def sort_part1():
    global hasil1
    hasil1 = shell_sort(part1)

def sort_part2():
    global hasil2
    hasil2 = shell_sort(part2)

thread1 = threading.Thread(target=sort_part1)
thread2 = threading.Thread(target=sort_part2)

tracemalloc.start()

start_multi = time.time()

thread1.start()
thread2.start()

thread1.join()
thread2.join()

merged = hasil1 + hasil2 # MENGGABUNGKAN HASIL

# Sorting akhir
multi_sorted = shell_sort(merged)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

improvement = (
    (single_time - multi_time) / single_time
) * 100


memory_usage_mb = peak_multi / (1024 * 1024) # rumus untuk mengubah byte ke megabyte
                                                # Memory Usage (MB) = Peak Memory (Byte) 1024 * 1024

# HASIL SORTING
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nHASIL SORTING SHELL SORT:\n")
display(df_sorted)

performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Shell Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 6)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 6)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 4)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")
display(performance_df)